In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
#pip install xlrd

In [3]:
#Download the file
df = pd.read_excel("GSAF5.xls")
df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,10th January,2026.0,Unprovoked,Australia,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,...,Unknown,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8th January,2026.0,Unprovoked,US Virgin Islands,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,...,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3rd January,2026.0,Unprovoked,New Caledonia,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,...,Unknown,Andy Currie: Province Sud:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,...,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,...,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
#Eliminar columnas por nombre 
columns_to_drop = { 
    'pdf',
    'href',
    'href formula',
    'Case Number',
    'Case Number.1',
    'original order',
    'Unnamed: 21',
    'Unnamed: 22'
}
df = df.drop(columns=columns_to_drop)

In [5]:
df.shape

(7065, 15)

### Column Type

In [6]:
#Looks Type 
df['Type'].unique()

array(['Unprovoked', 'Provoked', 'Questionable', 'unprovoked',
       ' Provoked', 'Watercraft', 'Sea Disaster', nan, '?', 'Unconfirmed',
       'Unverified', 'Invalid', 'Under investigation', 'Boat'],
      dtype=object)

In [7]:
#Clean type
df['Type'] = df['Type'].astype(str).str.strip().str.capitalize()

#Unificar valores raros a 'Unkown
df['Type'] = df['Type'].replace({
    'Unverified': 'Unknown',
    'Unconfirmed': 'Unknown',
    '?': 'Unknown',
    'Nan': 'Unknown',         
    'Invalid': 'Unknown',
    'Under investigation': 'Unknown',
    'Questionable' : 'Unknown'
})

#unificar 'Unprovoked' y 'Provoked' (ya los capitalizamos)
df['Type'] = df['Type'].replace({'Unprovoked': 'Unprovoked', 'Provoked': 'Provoked'})

#otros tipos 
df['Type'] = df['Type'].replace({'Watercraft': 'Other', 'Sea disaster': 'Other', 'Boat': 'Other'})

In [8]:
#Los cambios se hicieron efectivos 
df['Type'].unique()

array(['Unprovoked', 'Provoked', 'Unknown', 'Other'], dtype=object)

### Column Country

In [9]:
df['Country'].unique()

array(['Australia', 'US Virgin Islands', 'New Caledonia', 'USA',
       'French Polynesia', 'Samoa', 'Columbia', 'Costa Rica', 'Bahamas',
       'Puerto Rico', 'Spain', 'Canary Islands', 'South Africa',
       'Vanuatu', 'Jamaica', 'Israel', 'Mexico', 'Maldives',
       'Philippines', 'Turks and Caicos', 'Mozambique', 'Egypt',
       'Thailand', 'New Zealand', 'Hawaii', 'Honduras', 'Indonesia',
       'Morocco', 'Belize', 'Maldive Islands', 'Tobago', 'AUSTRALIA',
       'INDIA', 'TRINIDAD', 'BAHAMAS', 'SOUTH AFRICA', 'MEXICO',
       'NEW ZEALAND', 'EGYPT', 'BELIZE', 'PHILIPPINES', 'Coral Sea',
       'SPAIN', 'PORTUGAL', 'SAMOA', 'COLOMBIA', 'ECUADOR',
       'FRENCH POLYNESIA', 'NEW CALEDONIA', 'TURKS and CaICOS', 'CUBA',
       'BRAZIL', 'SEYCHELLES', 'ARGENTINA', 'FIJI', 'MeXICO', 'ENGLAND',
       'JAPAN', 'INDONESIA', 'JAMAICA', 'MALDIVES', 'THAILAND',
       'COLUMBIA', 'COSTA RICA', 'British Overseas Territory', 'CANADA',
       'JORDAN', 'ST KITTS / NEVIS', 'ST MARTIN', 'PAPUA

In [10]:
df['Country'].isnull().sum()

50

In [11]:
#unificando para que todo sea mayuscula o minusculas y quitando espacio en blanco 
df['Country'] = df['Country'].str.upper().str.strip()

In [12]:
#Diccionario de reemplaszos para unificar variantes:
replacements = {
    'AUSTRALIA': 'AUSTRALIA',
    'USA': 'USA',
    'US VIRGIN ISLANDS': 'USA',
    'UNITED STATES': 'USA',
    'NEW ZEALAND': 'NEW ZEALAND',
    'FIJI': 'FIJI',
    'COLOMBIA': 'COLOMBIA',
    'COLUMBIA': 'COLOMBIA',
    'MEXICO': 'MEXICO',
    'MALDIVES': 'MALDIVES',
    'MALDIVE ISLANDS': 'MALDIVES',
    'TURKS AND CAICOS': 'TURKS & CAICOS',
    'TURKS & CAICOS': 'TURKS & CAICOS',
    'PORTUGAL': 'PORTUGAL',
    'SPAIN': 'SPAIN',
    'UK': 'UNITED KINGDOM',
    'UNITED KINGDOM': 'UNITED KINGDOM',
    'ENGLAND': 'UNITED KINGDOM',
    'FRANCE': 'FRANCE',
    'GERMANY': 'GERMANY',
    'ITALY': 'ITALY',
    'JAPAN': 'JAPAN',
    'INDONESIA': 'INDONESIA',
    'THAILAND': 'THAILAND',
    'PHILIPPINES': 'PHILIPPINES',
    'BRAZIL': 'BRAZIL',
    'ARGENTINA': 'ARGENTINA',
    'SOUTH AFRICA': 'SOUTH AFRICA',
    'SAUDI ARABIA': 'SAUDI ARABIA',
    'EGYPT': 'EGYPT',
    'ISRAEL': 'ISRAEL',
    'CANADA': 'CANADA',
    'CEYLON': 'SRI LANKA',
    'CEYLON (SRI LANKA)': 'SRI LANKA', 

    # Océanos y mares los podemos poner como "OCEAN"
    'MEDITERRANEAN SEA': 'OCEAN',
    'ATLANTIC OCEAN': 'OCEAN',
    'TASMAN SEA': 'OCEAN',
    'RED SEA?': 'OCEAN',
    'CORAL SEA': 'OCEAN',
    'INDIAN OCEAN?': 'OCEAN',
    'BETWEEN PORTUGAL & INDIA': 'OCEAN',
    
    # Continentes o zonas grandes
    'COAST OF AFRICA': 'AFRICA',
    
    # Territorios pequeños que se pueden dejar como están o agrupar
    'ST MARTIN': 'ST MARTIN',
    'ST. MARTIN': 'ST MARTIN',
    'TRINIDAD & TOBAGO': 'TRINIDAD & TOBAGO',

  
    # Nan o vacíos
    'BRITISH NEW GUINEA': 'UNKNOWN',
    'ANDAMAN ISLANDS' : 'UNKNOWN',
    'COOK ISLANDS': 'UNKNOWN',
    'ROATAN': 'UNKNOWN',
    None: 'UNKNOWN',
    '': 'UNKNOWN',
    'AFRICA' : 'REGION',
    'ASIA?' : 'REGION',
    'KOREA': 'UNKNOWN',
    'EQUATORIAL GUINEA / CAMEROON': 'UNKNOWN'
}


df['Country'] = df['Country'].replace(replacements)

### Hipotesis 2: Mueren mas hombres que mujeres?

In [13]:
df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source
0,10th January,2026.0,Unprovoked,AUSTRALIA,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,puncture mark to left thumb,N,0540hrs,Unknown,Bob Myatt GSAF
1,8th January,2026.0,Unprovoked,USA,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,Left arm torn off in the attack below the elbow,Y,1628hrs,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com
2,3rd January,2026.0,Unprovoked,NEW CALEDONIA,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,Injuries to upper limbs,N,?,Unknown,Andy Currie: Province Sud:
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,Taken by shark body recovered with multiple in...,Y,1200hrs,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,Hand Injury,N,0800hrs,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7065 entries, 0 to 7064
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       7065 non-null   object 
 1   Year       7063 non-null   float64
 2   Type       7065 non-null   object 
 3   Country    7065 non-null   object 
 4   State      6578 non-null   object 
 5   Location   6498 non-null   object 
 6   Activity   6480 non-null   object 
 7   Name       6846 non-null   object 
 8   Sex        6486 non-null   object 
 9   Age        4070 non-null   object 
 10  Injury     7030 non-null   object 
 11  Fatal Y/N  6504 non-null   object 
 12  Time       3538 non-null   object 
 13  Species    3934 non-null   object 
 14  Source     7045 non-null   object 
dtypes: float64(1), object(14)
memory usage: 828.1+ KB


In [15]:
df['Sex'].unique()

array(['M', 'F', 'F ', 'M ', nan, ' M', 'm', 'lli', 'M x 2', 'N', '.'],
      dtype=object)

#### Eliminacion de Nulos de ['Sex']

In [16]:
data_sex = df[['Sex']]
data_sex

,Sex
0,M
1,F
2,M
3,F
4,M
...,...
7060,M
7061,M
7062,M
7063,M


In [17]:
sin_vacios = data_sex.dropna(subset=['Sex'])

In [18]:
print(data_sex.isnull().sum())

Sex    579
dtype: int64


data_sex.info()

In [19]:
sin_vacios.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6486 entries, 0 to 7064
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Sex     6486 non-null   object
dtypes: object(1)
memory usage: 101.3+ KB


In [20]:
sin_vacios['Sex'] = sin_vacios['Sex'].str.strip().str.upper()
sin_vacios['Sex'].unique()

/var/folders/m_/06pskt392pn6qvkq92krm8xm0000gn/T/ipykernel_49476/3308651882.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sin_vacios['Sex'] = sin_vacios['Sex'].str.strip().str.upper()


array(['M', 'F', 'LLI', 'M X 2', 'N', '.'], dtype=object)

In [21]:
#Unificar valores raros a 'Unkown
sin_vacios['Sex'] = sin_vacios['Sex'].replace({
    'M X 2': 'Unknown',
    'LLI': 'Unknown',
    'N': 'Unknown',
    '.': 'Unknown'       
})


/var/folders/m_/06pskt392pn6qvkq92krm8xm0000gn/T/ipykernel_49476/3112342755.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sin_vacios['Sex'] = sin_vacios['Sex'].replace({


In [22]:
sin_vacios['Sex'].unique()

array(['M', 'F', 'Unknown'], dtype=object)

In [23]:
sin_vacios.describe().T

,count,unique,top,freq
Sex,6486,3,M,5671


#### Hipotesis 2: Mueren mas gente <= 40 años 

In [24]:
age_df = df[['Age']]
age_df

,Age
0,?
1,56
2,?
3,55
4,?
...,...
7060,NaN
7061,NaN
7062,NaN
7063,NaN


In [25]:
age_df['Age'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 7065 entries, 0 to 7064
Series name: Age
Non-Null Count  Dtype 
--------------  ----- 
4070 non-null   object
dtypes: object(1)
memory usage: 55.3+ KB


In [26]:
age_df['Age'].unique()

array(['?', '56', '55', '24', '26', '25', '61', '40', '13', '14', '50+',
       '54', '48', '57', '8', '63', '9', '39', '19', '7', '85', '69',
       '18', '66', '21', '37', '16', '20', '12', '42', '45', '30', '30+',
       '40+', '29', 35, 58, 29, 24, 20, 55, 17, 12, 37, 36, 23, 40, 28,
       69, 48, '60+', 57, 45, 61, 27, 38, 16, 68, 33, 30, 15, 41, 14, 43,
       26, 'Middle age', 18, 21, 49, 25, 46, 19, 65, 64, nan, '11', '46',
       '32', '10', '64', '62', '22', '15', '52', '44', '47', '59', '50',
       '34', '38', '30s', '20/30', '35', '65', '20s', '77', '60', '49',
       '!2', '73', '50s', '58', '67', '17', '6', '41', '53', '68', '43',
       '51', '31', 39, 51, 10, 13, 60, '40s', 62, 'teen', 8, 22, 32, 56,
       'Teen', 42, 50, 'M', 9, 31, 11, 34, '!6', '!!', 47, 7, 71, 59, 53,
       54, 75, '45 and 15', 73, 52, 70, 4, 63, 44, '28 & 22',
       '22, 57, 31', '60s', "20's", 67, 74, '9 & 60', 'a minor', 6, 3, 82,
       '40?', 66, 72, '23', '36', '71', '70', '18 months', '2

In [28]:
# 1. Extraer solo números y dejar NaN donde no hay número
age_df['Age'] = age_df['Age'].astype(str).str.extract(r'(\d+)')


/var/folders/m_/06pskt392pn6qvkq92krm8xm0000gn/T/ipykernel_49476/1170685890.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  age_df['Age'] = age_df['Age'].astype(str).str.extract(r'(\d+)')


In [29]:
# 2. Convertir a float o int
age_df['Age'] = age_df['Age'].astype(float)

/var/folders/m_/06pskt392pn6qvkq92krm8xm0000gn/T/ipykernel_49476/2450266902.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  age_df['Age'] = age_df['Age'].astype(float)


In [32]:
# 3. Eliminar filas que todavía son NaN
age_df = age_df.dropna(subset=['Age'])

In [34]:
# 4. Convertir a int si quieres
age_df['Age'] = age_df['Age'].astype(int)

In [36]:
age_df['Age'].describe()

count    4020.000000
mean       28.163682
std        14.677832
min         1.000000
25%        17.000000
50%        24.000000
75%        37.000000
max        87.000000
Name: Age, dtype: float64

#### Conclusiones: